# Chinese Character Database — Creation Notebook

**Author:** Robert W Cellucci  
**Project:** CJK (Chinese-Japanese-Korean) SQLite Character Database  

This notebook builds the SQLite database from raw source files. The companion analysis notebook queries the finished database.

The database is structured around **Unicode codepoints** as the universal primary key. Every CJK character maps to exactly one codepoint, making codepoints a stable, encoding-agnostic anchor for all foreign-key relationships across tables.

---
## Data Sources
| Source | Coverage | Format |
|---|---|---|
| **Unihan** (Unicode Consortium) | Pan-CJK readings, variants, dictionary refs | TSV |
| **Taiwan MoE** (4808-character list) | Traditional Chinese education standard | TSV (pre-processed) |
| **HSK 3.0** | Mainland Chinese proficiency exam characters | CSV |
| **Heisig RTK** | Western learner kanji sequence (5th & 6th ed.) | CSV |

> **Note:** This is the *creation* notebook. Run all cells top-to-bottom on a clean environment. 
> If the database file already exists, `CREATE TABLE IF NOT EXISTS` guards prevent duplicate table creation, 
> and `INSERT OR IGNORE` prevents duplicate row insertion, so re-runs are safe.


## Import Statements

Only two top-level imports are needed:
- `pandas` — in-memory data wrangling and ETL staging before DB writes
- `sqlite3` — Python's built-in SQLite driver; no installation required

In [137]:
import pandas as pd    # DataFrame-based ETL; used as a staging layer before every DB write
import sqlite3          # CPython built-in; no pip install needed

In [138]:
# DATA VALIDATION: source File Existence Check
# Before any parsing begins, confirm every expected source file is present on disk.

import os

required_sources = [
    'sources/Unihan_Readings.tsv',
    'sources/Unihan_DictionaryIndices.tsv',
    'sources/Unihan_OtherMappings.tsv',
    'sources/Unihan_DictionaryLikeData.tsv',
    'sources/Unihan_IRGSources.tsv',
    'sources/Unihan_Variants.tsv',
    'sources/moe_4808_unicode.tsv',
    'sources/hsk_3.0_characters.csv',
    'sources/heisig-kanjis.csv',
]

missing = [f for f in required_sources if not os.path.exists(f)]

if missing:
    print('Missing source files — resolve before proceeding:')
    for f in missing:
        print(f'   {f}')
    raise FileNotFoundError(
        f'{len(missing)} required source file(s) not found. See list above.'
    )
else:
    print(f'All {len(required_sources)} required source files found.')

All 9 required source files found.


## Core Relational Table: `Character_Table`

This is the **spine of the entire schema**. Every other table holds a foreign key back to `character_table.codepoint`.

The scope of the project is defined here: only characters whose codepoints appear in this table will appear anywhere else in the database. Characters are inserted via `INSERT OR IGNORE` from each source (Korean MoE, Taiwan MoE, HSK, Joyo/Jinmeiyo, Heisig) so codepoints accumulate without duplication as new sources are added.

Columns:
- `codepoint` — Unicode scalar value stored as INTEGER (e.g. `0x4E2D` → `20013` for 中). PRIMARY KEY.
- `literal` — The actual character glyph stored as TEXT, derived from `chr(codepoint)`. Redundant but convenient for human-readable queries.


In [139]:
# `with sqlite3.connect(...) as conn` opens a connection and auto-commits (or rolls back)
# when the block exits — no manual conn.close() needed.
#
# PRAGMA foreign_keys = ON  →  SQLite disables FK enforcement by default; this line enables it
#                               for the duration of the connection. Must be re-issued each time
#                               a new connection is opened — it does not persist in the file.
#
# CREATE TABLE IF NOT EXISTS  →  idempotent; safe to re-run without dropping existing data.
#
# CONSTRAINT syntax names the PK explicitly so error messages and EXPLAIN output are readable.

with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("""
    CREATE TABLE IF NOT EXISTS character_table (
        codepoint INTEGER,
        literal TEXT,
        CONSTRAINT pk_character_table PRIMARY KEY (codepoint)
    )
""")
    conn.commit()

## Data Source 1: Unihan TSV Files

The [Unihan Database](https://www.unicode.org/charts/unihan.html) is the Unicode Consortium's authoritative dataset for all CJK Unified Ideographs. It is distributed as a set of tab-separated files, each covering a different category of properties:

| File | Contents |
|---|---|
| `Unihan_Readings.tsv` | Mandarin, Cantonese, Korean, Vietnamese readings |
| `Unihan_DictionaryIndices.tsv` | Entry numbers in classical dictionaries (KangXi, Morohashi, Nelson…) |
| `Unihan_OtherMappings.tsv` | Joyo, Jinmeiyo, Korean MoE, and other standard list memberships |
| `Unihan_DictionaryLikeData.tsv` | Stroke counts, radical numbers, frequency data |
| `Unihan_IRGSources.tsv` | IRG (Ideographic Research Group) source references |
| `Unihan_Variants.tsv` | Simplified↔Traditional and other variant mappings |

Each file shares the same three-column schema: `codepoint` (e.g. `U+4E2D`), `field` (property name), `value`.
They are concatenated and then **pivoted** so each codepoint becomes one wide row.

> **Han Unification caveat:** Unicode merged visually similar characters from Chinese, Japanese, and Korean into single codepoints (Han Unification). Regional glyph variation is left to fonts, not to separate codepoints. This is a known limitation of the dataset and means some characters that look different in practice share a codepoint here.


In [140]:
# pd.concat stacks all six TSV files into one tall DataFrame with columns:
#   codepoint | field | value
# This 'long' format is the native shape of the Unihan files.
#
# sep='\t'        — TSV, not CSV
# comment='#'     — Unihan files begin with several lines of # metadata; skip them
# header=None     — the data rows have no header row
# names=[...]     — we supply the column names manually
# on_bad_lines='warn'  — malformed rows are logged but don't abort the parse
# ignore_index=True    — resets the row index after concatenation

unihan_raw_data = pd.concat([
    pd.read_csv(
        'sources/Unihan_Readings.tsv',
        sep='\t',
        comment='#',
        header=None,
        names=['codepoint', 'field', 'value'],
        on_bad_lines='warn'
    ),
    pd.read_csv(
        'sources/Unihan_DictionaryIndices.tsv',
        sep='\t',
        comment='#',
        header=None,
        names=['codepoint', 'field', 'value'],
        on_bad_lines='warn'
    ),
    pd.read_csv(
        'sources/Unihan_OtherMappings.tsv',
        sep='\t',
        comment='#',
        header=None,
        names=['codepoint', 'field', 'value'],
        on_bad_lines='warn'
    ),
     pd.read_csv(
        'sources/Unihan_DictionaryLikeData.tsv',
        sep='\t',
        comment='#',
        header=None,
        names=['codepoint', 'field', 'value'],
        on_bad_lines='warn'
    ),
    pd.read_csv(
        'sources/Unihan_IRGSources.tsv',
        sep='\t',
        comment='#',
        header=None,
        names=['codepoint', 'field', 'value'],
        on_bad_lines='warn'
    ),
    pd.read_csv(
        'sources/Unihan_Variants.tsv',
        sep='\t',
        comment='#',
        header=None,
        names=['codepoint', 'field', 'value'],
        on_bad_lines='warn'
    )
], ignore_index=True)

In [141]:
# pivot_table reshapes the long (codepoint, field, value) DataFrame into a wide one
# where each unique 'field' value becomes its own column.
# Result shape: one row per codepoint, one column per Unihan property (kMandarin, kJoyoKanji, etc.)
#
# aggfunc='first'  → some codepoints have multiple values for the same field (e.g. variant forms);
#                    we keep only the first. This is a deliberate simplification.
#
# reset_index()    → pivot_table makes 'codepoint' the index; this promotes it back to a column.
#
# .columns.name = None  → pivot_table sets columns.name to 'field' as a label artifact;
#                          clearing it keeps the DataFrame display clean.
#
# The lambda converts 'U+4E2D' → 20013 (decimal int) using int(..., 16) for base-16 parsing.
# This aligns the codepoint format with ord() values used everywhere else in the project.

unihan_data = unihan_raw_data.pivot_table(
    index='codepoint',
    columns='field',
    values='value',
    aggfunc='first'
).reset_index()
unihan_data.columns.name = None

unihan_data['codepoint'] = unihan_data['codepoint'].apply(lambda x: int(x.replace('U+', ''), 16))
display(unihan_data)

,codepoint,kAlternateTotalStrokes,kBigFive,kCCCII,kCNS1986,kCNS1992,kCangjie,kCantonese,kCheungBauer,kCheungBauerIndex,...,kTaiwanTelegraph,kTang,kTotalStrokes,kTraditionalVariant,kUnihanCore2020,kVietnamese,kXHC1983,kXerox,kZVariant,kZhuang
0,131072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,131073,NaN,NaN,NaN,NaN,NaN,NaN,cat1,NaN,NaN,...,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,131074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,131075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,131076,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102993,64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102994,64214,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102995,64215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102996,64216,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Unihan Korean Tags: Ministry of Education & Korean Name Lists

Unihan provides two Korean government-sanctioned character lists via its `kKorean*` fields:

- **`kKoreanEducationHanja`** — Characters designated by the Korean Ministry of Education for secondary school instruction (교육용 한자). This is the primary Korean scope list for the database.
- **`kKoreanName`** — Characters approved by the Korean government for use in personal names (인명용 한자).

Characters appearing in either list are added to `Character_Table` and tagged in `Character_Source`.
Note that a character can appear in both lists (education *and* name use), which is why `how='all'` is used in `dropna`, we keep any row with *at least one* non-null tag.


In [142]:
# Slice just the two korean government tag columns from the wide Unihan DataFrame.
korean_unihan = unihan_data[['codepoint', 'kKoreanEducationHanja','kKoreanName']].copy()

# dropna(how='all')  → keep rows where at least one of the two columns is non-null.
korean_unihan.dropna(subset=['kKoreanEducationHanja','kKoreanName'], how='all',inplace=True)

# The lambda chr(x) converts an integer codepoint back to the Unicode character glyph.
korean_unihan['literal'] = korean_unihan['codepoint'].apply(lambda x: chr(x))

# pop/insert is used to reorder columns so 'literal' appears immediately after 'codepoint'.
korean_literals = korean_unihan.pop('literal')
korean_unihan.insert(1, 'literal', korean_literals)

display(korean_unihan)

,codepoint,literal,kKoreanEducationHanja,kKoreanName
215,131287,𠃗,NaN,2015
300,131372,𠄬,NaN,2015
1482,132554,𠗊,NaN,2015
2436,133508,𠦄,NaN,2015
4437,135509,𡅕,NaN,2015
...,...,...,...,...
102426,40860,龜,2007,2015
102427,40861,龝,NaN,2015
102430,40864,龠,NaN,2015
102432,40866,龢,NaN,2015


In [143]:
# Populating korean Characters into Character_Table
#
# Pattern used throughout this notebook for safe, idempotent DB inserts:
#   1. Write the staging DataFrame to a temporary table '_temp_char' (replacing if it exists)
#   2. INSERT OR IGNORE into the real table, which silently skips codepoints already present
#   3. DROP the temp table immediately to keep the schema clean
#   4. Commit and display the current state of the target table for verification
#
# This staging approach is necessary because pandas .to_sql() does not support
# INSERT OR IGNORE directly; it can only do INSERT (fail on conflict) or REPLACE (overwrite).
# The temp table workaround gives us conflict-safe upsert semantics.

korean_characters = korean_unihan[['codepoint','literal']].copy()

with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")  
    korean_characters.to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO character_table
        SELECT * FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")    
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM character_table", conn))

,codepoint,literal
0,13466,㒚
1,13527,㓗
2,13589,㔕
3,13601,㔡
4,13823,㗿
...,...,...
8089,173668,𪙤
8090,180501,𬄕
8091,182227,𬟓
8092,189801,𮕩


In [144]:
# Build a source-provenance record for the korean Education list.
# We tag each codepoint with 'source' and 'source_region' rather than keeping
# the raw kKoreanEducationHanja value (which is just a grade/level string we don't need here).
# The actual column is dropped after the tags are set.

korean_ed = unihan_data[['codepoint','kKoreanEducationHanja']].copy()
korean_ed = korean_ed.dropna(subset=['kKoreanEducationHanja'])
korean_ed['source'] = 'KoreanEducationHanja'
korean_ed['source_region'] = 'Korea'
korean_ed = korean_ed.drop('kKoreanEducationHanja',axis=1)
display(korean_ed)


,codepoint,source,source_region
58693,189801,KoreanEducationHanja,Korea
81534,19968,KoreanEducationHanja,Korea
81535,19969,KoreanEducationHanja,Korea
81537,19971,KoreanEducationHanja,Korea
81542,19976,KoreanEducationHanja,Korea
...,...,...,...
102329,40763,KoreanEducationHanja,Korea
102344,40778,KoreanEducationHanja,Korea
102352,40786,KoreanEducationHanja,Korea
102411,40845,KoreanEducationHanja,Korea


In [145]:
# Same provenance-tagging pattern as korean_ed above, applied to the Name Hanja list.
# These are characters legally permitted in korean personal names which is a separate
# government-maintained list from the education list.

korean_names = unihan_data[['codepoint','kKoreanName']].copy()
korean_names = korean_names.dropna(subset=['kKoreanName'])
korean_names['source'] = 'KoreanName'
korean_names['source_region'] = 'Korea'
korean_names = korean_names.drop('kKoreanName',axis=1)
display(korean_names)


,codepoint,source,source_region
215,131287,KoreanName,Korea
300,131372,KoreanName,Korea
1482,132554,KoreanName,Korea
2436,133508,KoreanName,Korea
4437,135509,KoreanName,Korea
...,...,...,...
102426,40860,KoreanName,Korea
102427,40861,KoreanName,Korea
102430,40864,KoreanName,Korea
102432,40866,KoreanName,Korea


## Unihan Japanese Tags: Jōyō and Jinmeiyō Kanji

Unihan also provides Japan's two government-mandated kanji lists:

- **`kJoyoKanji`** — The [Jōyō kanji](https://en.wikipedia.org/wiki/J%C5%8Dy%C5%8D_kanji) (常用漢字), 2136 characters designated for general daily use. All Japanese public school students are expected to know these by graduation.
- **`kJinmeiyoKanji`** — The [Jinmeiyō kanji](https://en.wikipedia.org/wiki/Jinmeiy%C5%8D_kanji) (人名用漢字), characters approved for use in registered personal names but not in the Jōyō set.

These two lists together define the practical Japanese scope of the database. The Heisig RTK data (below) supplements this with a learner-centric ordering.


In [146]:
# Tag Joyo kanji codepoints with their source provenance.
# The kJoyoKanji column value (e.g. '2010') indicates the revision year of the list
# in which the character was added; we discard this value and keep only the presence flag
# by dropping the column after tagging.

japanese_joyo = unihan_data[['codepoint','kJoyoKanji']].copy()
japanese_joyo = japanese_joyo.dropna(subset=['kJoyoKanji'])
japanese_joyo = japanese_joyo[~japanese_joyo['kJoyoKanji'].astype(str).str.startswith('U+')]
japanese_joyo['source'] = 'Joyo'
japanese_joyo['source_region'] = 'Japan'
japanese_joyo = japanese_joyo.drop('kJoyoKanji', axis=1)
display(japanese_joyo)


,codepoint,source,source_region
2975,134047,Joyo,Japan
81534,19968,Joyo,Japan
81535,19969,Joyo,Japan
81537,19971,Joyo,Japan
81541,19975,Joyo,Japan
...,...,...,...
102224,40658,Joyo,Japan
102231,40665,Joyo,Japan
102289,40723,Joyo,Japan
102329,40763,Joyo,Japan


In [147]:
# Identical pattern to japanese_joyo above.
# kJinmeiyoKanji values encode the source list revision; presence alone is what matters.

japanese_jinmeiyo = unihan_data[['codepoint','kJinmeiyoKanji']].copy()
japanese_jinmeiyo = japanese_jinmeiyo.dropna(subset=['kJinmeiyoKanji'])
japanese_jinmeiyo['source'] = 'Jinmeiyo'
japanese_jinmeiyo['source_region'] = 'Japan'
japanese_jinmeiyo = japanese_jinmeiyo.drop('kJinmeiyoKanji',axis=1)
display(japanese_jinmeiyo)


,codepoint,source,source_region
81551,19985,Jinmeiyo,Japan
81564,19998,Jinmeiyo,Japan
81601,20035,Jinmeiyo,Japan
81609,20043,Jinmeiyo,Japan
81612,20046,Jinmeiyo,Japan
...,...,...,...
102882,64100,Jinmeiyo,Japan
102883,64101,Jinmeiyo,Japan
102885,64103,Jinmeiyo,Japan
102886,64104,Jinmeiyo,Japan


## Data Source: Taiwan Ministry of Education — 4808-Character Standard List

Taiwan's MoE publishes an official list of 4808 standard-form Traditional Chinese characters (國字標準字體). 
The source PDF was converted to a TSV using AI-assisted extraction (Thanks Claude!), then cleaned into the format used here.

This list defines the Traditional Chinese scope of the database. Note that Taiwan characters 
will often overlap with Joyo/Jinmeiyo (Japan also uses traditional forms for many kanji) 
and with HSK traditional forms. `INSERT OR IGNORE` handles these overlaps.

Column notes on the raw TSV:
- `seq` / `moe_id` — MoE internal sequence and ID numbers; not needed in the DB schema
- `unicode_hex` — hex codepoint string (e.g. `4E2D`); redundant once we have `unicode_int`
- `unicode_int` — decimal codepoint; this becomes our `codepoint` column
- `char` — the literal character glyph; becomes `literal`


In [148]:
# Read the pre-processed Taiwan MoE TSV.
# Rename unicode_int → codepoint to match the project's unified column name.
# Drop the metadata columns (seq, moe_id, unicode_hex, unicode_int) that aren't needed in the schema.
# pop/insert reorders so 'char' (renamed to 'literal' by convention downstream) is column 1.

taiwan_data = pd.read_csv('sources/moe_4808_unicode.tsv', sep='\t', on_bad_lines='warn')
taiwan_data['codepoint'] = taiwan_data['unicode_int']
taiwan_data = taiwan_data.drop(['seq', 'moe_id','unicode_hex','unicode_int'],axis=1)
col = taiwan_data.pop('char')
taiwan_data.insert(1,'literal',col)
display(taiwan_data)


,codepoint,literal
0,19968,一
1,19969,丁
2,19971,七
3,19977,三
4,19979,下
...,...,...
4803,40823,齷
4804,40818,齲
4805,40845,龍
4806,40852,龔


In [149]:
# Populating Taiwanese Characters into Character_Table
# Uses the standard temp-table INSERT OR IGNORE pattern (see Cell 11 for full explanation).
# Only codepoint and literal are written; the other taiwan_data columns are source metadata.

with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")  
    taiwan_data[['codepoint','literal']].to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO character_table
        SELECT * FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")    
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM character_table", conn))

,codepoint,literal
0,13466,㒚
1,13527,㓗
2,13589,㔕
3,13601,㔡
4,13823,㗿
...,...,...
8528,173668,𪙤
8529,180501,𬄕
8530,182227,𬟓
8531,189801,𮕩


In [150]:
# Build provenance record for the Taiwan MoE list.
# 'literal' is dropped here because Character_Source tracks codepoint+source, not the glyph.
# The glyph is already stored in Character_Table.

taiwan_ed = taiwan_data.copy()
taiwan_ed['source'] = 'TaiwanEducationHanzi'
taiwan_ed['source_region'] = 'Taiwan'
taiwan_ed = taiwan_ed.drop('literal',axis=1)
display(taiwan_ed)

,codepoint,source,source_region
0,19968,TaiwanEducationHanzi,Taiwan
1,19969,TaiwanEducationHanzi,Taiwan
2,19971,TaiwanEducationHanzi,Taiwan
3,19977,TaiwanEducationHanzi,Taiwan
4,19979,TaiwanEducationHanzi,Taiwan
...,...,...,...
4803,40823,TaiwanEducationHanzi,Taiwan
4804,40818,TaiwanEducationHanzi,Taiwan
4805,40845,TaiwanEducationHanzi,Taiwan
4806,40852,TaiwanEducationHanzi,Taiwan


## Data Source: HSK 3.0 Character Lists

The **Hanyu Shuiping Kaoshi** (汉语水平考试, HSK) is the PRC's official Mandarin proficiency exam for non-native speakers. 
The 2021 'HSK 3.0' revision reorganized the exam into nine levels.

The CSV used here (sourced from a GitHub repository) contains HSK 3.0 vocabulary, with one row per simplified character and a companion column for the corresponding traditional form(s).

**Key complexity:** The simplified↔traditional mapping is **many-to-one**. A single simplified character 
can correspond to multiple traditional characters, and the traditional column may contain multiple glyphs 
as a single string (e.g. `'發髮'`). The next two cells explode this into one row per traditional codepoint.

Both the simplified and traditional codepoints are added to `Character_Table` and `Character_Source`, 
and the mapping itself is stored in `Trad_Simp_Map`.


In [151]:
# Read HSK 3.0 data. The CSV has one row per vocabulary item.
hsk_data = pd.read_csv('sources/hsk_3.0_characters.csv', on_bad_lines='warn')

# ord(x)  →  returns the Unicode codepoint of a single character.
#             We apply it to the 'hanzi_sc' (simplified) column to get integer codepoints.
#             This assumes hanzi_sc is always a single character valid for this dataset.
hsk_data['s_codepoint'] = hsk_data['hanzi_sc'].apply(lambda x: ord(x))

# pop/insert pattern moves S_codepoint to position 0 (leftmost column) for readability.
col = hsk_data.pop('s_codepoint')
hsk_data.insert(0, 's_codepoint', col)
display(hsk_data)


,s_codepoint,hanzi_sc,hanzi_trad,pinyin,pinyin_style2,level,level_zh,cc_cedict_definitions
0,30340,的,的,de,de,1,初等|一级,(bound form) bull's-eye; target
1,20102,了,了瞭,le,le,1,初等|一级,unofficial variant of 瞭[liao4]
2,22312,在,在,zài,zai4,1,初等|一级,to exist; to be alive/(of sb or sth) to be (lo...
3,26159,是,是,shì,shi4,1,初等|一级,variant of 是[shi4]
4,25105,我,我,wǒ,wo3,1,初等|一级,I; me; my
...,...,...,...,...,...,...,...,...
2995,37011,邓,鄧,dèng,deng4,9,高等,surname Deng
2996,28142,淮,淮,huái,huai2,9,高等,name of a river
2997,34945,袁,袁,yuán,yuan2,9,高等,long robe (old)
2998,31908,粤,粵,yuè,yue4,9,高等,Cantonese/short name for Guangdong 廣東|广东[Guang...


In [152]:
# Explode the traditional character column from a multi-character string into individual rows.
#
# The 'hanzi_trad' column may contain strings like '發髮' (two characters) — one simplified
# character mapping to two distinct traditional forms.
#
# Step-by-step:
#   list(x)           →  split string into a list of individual characters: '發髮' → ['發','髮']
#                         isinstance guard handles NaN (non-string) rows, returning [] instead.
#   [ord(c) for c in] →  convert each character to its codepoint integer
#   .explode([...])   →  one row per element; both 'hanzi_trad_list' and 'T_codepoint'
#                         are exploded in parallel so row alignment is preserved.
#                         Parallel explode requires pandas >= 1.3.
#   drop_duplicates   →  some traditional characters appear for multiple simplified entries;
#                         keep only the first occurrence to avoid duplicate PKs in the DB.
#   .copy()           →  avoid SettingWithCopyWarning on the deduplicated slice.

hsk_data_t = hsk_data[['s_codepoint','hanzi_sc', 'hanzi_trad']].copy()

hsk_data_t['hanzi_trad_list'] = hsk_data_t['hanzi_trad'].apply(
    lambda x: list(x) if isinstance(x, str) else []
)

hsk_data_t['t_codepoint'] = hsk_data_t['hanzi_trad_list'].apply(
    lambda chars: [ord(c) for c in chars]
)

# Explode so each trad character gets its own row
hsk_data_t = hsk_data_t.explode(['hanzi_trad_list', 't_codepoint']).reset_index(drop=True)
hsk_data_t = hsk_data_t.drop(['hanzi_trad'],axis=1)

col = hsk_data_t.pop('t_codepoint')
hsk_data_t.insert(0, 't_codepoint', col)

hsk_data_t['t_codepoint'] = hsk_data_t['t_codepoint'].astype(int)
hsk_data_t_clean = hsk_data_t.drop_duplicates(subset='t_codepoint', keep='first')
hsk_data_t_clean = hsk_data_t_clean.copy()
display(hsk_data_t_clean)

,t_codepoint,s_codepoint,hanzi_sc,hanzi_trad_list
0,30340,30340,的,的
1,20102,20102,了,了
2,30637,20102,了,瞭
3,22312,22312,在,在
4,26159,26159,是,是
...,...,...,...,...
3160,37159,37011,邓,鄧
3161,28142,28142,淮,淮
3162,34945,34945,袁,袁
3163,31925,31908,粤,粵


In [153]:
hsk_trad = hsk_data_t_clean[['t_codepoint','hanzi_trad_list']].copy()
hsk_trad = hsk_trad.rename(columns={'t_codepoint': 'codepoint'})
hsk_trad = hsk_trad.rename(columns={'hanzi_trad_list': 'literal'})
display(hsk_trad)

,codepoint,literal
0,30340,的
1,20102,了
2,30637,瞭
3,22312,在
4,26159,是
...,...,...
3160,37159,鄧
3161,28142,淮
3162,34945,袁
3163,31925,粵


In [154]:
# Populating Traditional Mainland Characters into Character_Table
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")  
    hsk_trad[['codepoint','literal']].to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO character_table
        SELECT * FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")    
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM character_table", conn))

,codepoint,literal
0,13466,㒚
1,13527,㓗
2,13589,㔕
3,13601,㔡
4,13823,㗿
...,...,...
8562,173668,𪙤
8563,180501,𬄕
8564,182227,𬟓
8565,189801,𮕩


In [155]:
hsk_trad = hsk_trad.drop('literal',axis=1)
hsk_trad['source'] = 'hsk_trad'
hsk_trad['source_region'] = 'China'
display(hsk_trad)

,codepoint,source,source_region
0,30340,hsk_trad,China
1,20102,hsk_trad,China
2,30637,hsk_trad,China
3,22312,hsk_trad,China
4,26159,hsk_trad,China
...,...,...,...
3160,37159,hsk_trad,China
3161,28142,hsk_trad,China
3162,34945,hsk_trad,China
3163,31925,hsk_trad,China


In [156]:
hsk_simp = hsk_data_t_clean[['s_codepoint','hanzi_sc']].copy()
hsk_simp = hsk_simp.rename(columns={'s_codepoint': 'codepoint'})
hsk_simp = hsk_simp.rename(columns={'hanzi_sc': 'literal'})

In [157]:
# Populating Simplifed Mainland Characters into Character_Table
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")  
    hsk_simp[['codepoint','literal']].to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO character_table
        SELECT * FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")    
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM character_table", conn))

,codepoint,literal
0,13466,㒚
1,13527,㓗
2,13589,㔕
3,13601,㔡
4,13823,㗿
...,...,...
9511,173668,𪙤
9512,180501,𬄕
9513,182227,𬟓
9514,189801,𮕩


In [158]:
# Build provenance record for HSK simplified characters.
# Source: hsk_data_t_clean, which was deduplicated on t_codepoint during the explode step.
# We further deduplicate on s_codepoint here because multiple traditional forms
# can map to the same simplified character — we only need one provenance row per
# simplified codepoint in character_source.
hsk_simp = hsk_data_t_clean[['s_codepoint']].drop_duplicates(subset='s_codepoint').copy()
hsk_simp = hsk_simp.rename(columns={'s_codepoint': 'codepoint'})
hsk_simp['source'] = 'hsk_simp'
hsk_simp['source_region'] = 'China'
display(hsk_simp)


,codepoint,source,source_region
0,30340,hsk_simp,China
1,20102,hsk_simp,China
3,22312,hsk_simp,China
4,26159,hsk_simp,China
5,25105,hsk_simp,China
...,...,...,...
3160,37011,hsk_simp,China
3161,28142,hsk_simp,China
3162,34945,hsk_simp,China
3163,31908,hsk_simp,China


## Data Source: Heisig *Remembering the Kanji* (RTK)

[James Heisig](https://en.wikipedia.org/wiki/Remembering_the_Kanji_and_Remembering_the_Hanzi)'s 
*Remembering the Kanji* is the dominant Western method for learning kanji via mnemonic stories 
tied to meaning (not pronunciation). First published in 1977, it is widely used alongside 
Anki SRS decks by English-speaking learners.

The CSV (sourced from GitHub) provides:
- `kanji` — the literal character
- `id_5th_ed` — Heisig's sequence number in the 5th edition
- `id_6th_ed` — sequence number in the 6th edition (adds ~200 characters)
- Additional metadata (keyword, stroke count, etc.) not used in this notebook

The 5th and 6th editions are treated as separate sources in `Character_Source` because 
they have different character sets and orderings. A character may appear in one edition 
but not the other.


In [159]:
# ord(x) converts the kanji glyph character to its integer codepoint.
# This is necessary because the CSV stores the literal character, but the schema uses integer codepoints as primary keys everywhere.
# pop/insert moves codepoint to position 0 for consistent column ordering.

heisig_data = pd.read_csv('sources/heisig-kanjis.csv', on_bad_lines='warn')
heisig_data['codepoint'] = heisig_data['kanji'].apply(lambda x: ord(x))
col = heisig_data.pop('codepoint')
heisig_data.insert(0, 'codepoint', col)
display(heisig_data)


,codepoint,kanji,id_5th_ed,id_6th_ed,keyword_5th_ed,keyword_6th_ed,components,on_reading,kun_reading,stroke_count,jlpt
0,19968,一,1.0,1.0,one,one,NaN,イチ; イツ,ひと-; ひと.つ,1.0,N5
1,20108,二,2.0,2.0,two,two,NaN,ニ; ジ,ふた; ふた.つ; ふたた.び,2.0,N5
2,19977,三,3.0,3.0,three,three,NaN,サン; ゾウ,み; み.つ; みっ.つ,3.0,N5
3,22235,四,4.0,4.0,four,four,pent in; human legs,シ,よ; よ.つ; よっ.つ; よん,5.0,N5
4,20116,五,5.0,5.0,five,five,NaN,ゴ,いつ; いつ.つ,4.0,N5
...,...,...,...,...,...,...,...,...,...,...,...
3034,35409,詑,2718.0,NaN,prevarication,NaN,NaN,NaN,NaN,NaN,NaN
3035,35535,諏,2722.0,NaN,advise,NaN,NaN,NaN,NaN,NaN,NaN
3036,36527,躯,2986.0,NaN,body (old),NaN,NaN,NaN,NaN,NaN,NaN
3037,37057,郁,2424.0,NaN,cultured,NaN,NaN,NaN,NaN,NaN,NaN


In [160]:
heisig5 = heisig_data[['codepoint','id_5th_ed']].copy()
heisig5['source'] =  'RememberingTheKanji5th'
heisig5['source_region'] = 'Japan'
heisig5 = heisig5.dropna(subset=['id_5th_ed'])
heisig5_source = heisig5.drop('id_5th_ed',axis=1)
display(heisig5_source)

,codepoint,source,source_region
0,19968,RememberingTheKanji5th,Japan
1,20108,RememberingTheKanji5th,Japan
2,19977,RememberingTheKanji5th,Japan
3,22235,RememberingTheKanji5th,Japan
4,20116,RememberingTheKanji5th,Japan
...,...,...,...
3034,35409,RememberingTheKanji5th,Japan
3035,35535,RememberingTheKanji5th,Japan
3036,36527,RememberingTheKanji5th,Japan
3037,37057,RememberingTheKanji5th,Japan


In [161]:
heisig6 = heisig_data[['codepoint','id_6th_ed']].copy()
heisig6['source'] =  'RememberingTheKanji6th'
heisig6['source_region'] = 'Japan'
heisig6 = heisig6.dropna(subset=['id_6th_ed'])
heisig6_source = heisig6.drop('id_6th_ed',axis=1)
display(heisig6_source)

,codepoint,source,source_region
0,19968,RememberingTheKanji6th,Japan
1,20108,RememberingTheKanji6th,Japan
2,19977,RememberingTheKanji6th,Japan
3,22235,RememberingTheKanji6th,Japan
4,20116,RememberingTheKanji6th,Japan
...,...,...,...
2995,22781,RememberingTheKanji6th,Japan
2996,36490,RememberingTheKanji6th,Japan
2997,39237,RememberingTheKanji6th,Japan
2998,22036,RememberingTheKanji6th,Japan


## Assembling the Japanese Character Set

The full Japanese scope is the **union** of:
- Jōyō kanji (from Unihan `kJoyoKanji`)
- Jinmeiyō kanji (from Unihan `kJinmeiyoKanji`)
- Heisig RTK kanji (5th and/or 6th edition)

An **outer join** is used between Unihan and Heisig so that kanji present in only one 
source are still included. This is intentional — Heisig includes some characters not on 
the government lists, and some Jōyō kanji may not be in Heisig.


In [162]:
j_set = pd.merge(japanese_joyo,japanese_jinmeiyo,on='codepoint',how='outer')

In [163]:
# Outer join Unihan Japanese set with Heisig data.
# how='outer'  →  include codepoints from either source, not just their intersection.
#                  Heisig-only kanji get NaN for kJoyoKanji/kJinmeiyoKanji; Joyo-only
#                  kanji get NaN for id_5th_ed/id_6th_ed. Both are still valid characters.
# The 'kanji' literal column from heisig_data_c is dropped because we re-derive
# the literal from the codepoint using chr() for consistency.

heisig_data_c = heisig_data[['codepoint','kanji','id_5th_ed','id_6th_ed']]
j_characters = pd.merge(j_set,heisig_data_c, on='codepoint',how='outer')
j_characters = j_characters.drop(['kanji'],axis=1)
j_characters['literal'] = j_characters['codepoint'].apply(lambda x: chr(x))
col = j_characters.pop('literal')
j_characters.insert(1,'literal',col)
display(j_characters)


,codepoint,literal,source_x,source_region_x,source_y,source_region_y,id_5th_ed,id_6th_ed
0,19968,一,Joyo,Japan,NaN,NaN,1.0,1.0
1,19969,丁,Joyo,Japan,NaN,NaN,91.0,95.0
2,19971,七,Joyo,Japan,NaN,NaN,7.0,7.0
3,19975,万,Joyo,Japan,NaN,NaN,64.0,68.0
4,19976,丈,Joyo,Japan,NaN,NaN,691.0,746.0
...,...,...,...,...,...,...,...,...
3320,64101,贈,NaN,NaN,Jinmeiyo,Japan,NaN,NaN
3321,64103,逸,NaN,NaN,Jinmeiyo,Japan,NaN,NaN
3322,64104,難,NaN,NaN,Jinmeiyo,Japan,NaN,NaN
3323,64105,響,NaN,NaN,Jinmeiyo,Japan,NaN,NaN


In [164]:
# Populating Japanese Characters into Character_Table
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")  
    j_characters[['codepoint','literal']].to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO character_table
        SELECT * FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")    
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM character_table", conn))

,codepoint,literal
0,13466,㒚
1,13527,㓗
2,13589,㔕
3,13601,㔡
4,13823,㗿
...,...,...
9835,173668,𪙤
9836,180501,𬄕
9837,182227,𬟓
9838,189801,𮕩


## Character Source Table

`Character_Source` is the **provenance table**. It records *which official list(s)* each character 
comes from and *which region* those lists originate from.

**Schema design note:** The PRIMARY KEY is `(codepoint, source)`, a composite key, because 
the same codepoint can legitimately appear in multiple sources (e.g. 水 is Jōyō *and* in Heisig *and* in HSK). 
A single-column PK on codepoint alone would wrongly deduplicate cross-regional characters.

The `Source` DataFrame assembled in the next cell is the vertical concatenation of all provenance 
records built throughout the earlier sections.


In [165]:
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS character_source (
            codepoint INTEGER,
            source TEXT,
            source_region TEXT,
            CONSTRAINT pk_codepoint PRIMARY KEY (codepoint,source),
            CONSTRAINT fk_codepoint FOREIGN KEY (codepoint)
            REFERENCES character_table (codepoint)
        )
    """)
    conn.commit()

In [166]:
# Concatenate all provenance DataFrames built throughout the notebook.
# Each sub-DataFrame has the same three columns: codepoint | source | source_region.
# ignore_index=True resets the index to avoid duplicate index values from the component DFs.
# The order here determines insert order into Character_Source, which has no effect on
# query results but can matter for debugging (earlier rows are easier to inspect).

source = pd.concat([
    hsk_trad,
    hsk_simp,
    taiwan_ed,
    japanese_joyo,
    japanese_jinmeiyo,
    heisig6_source,
    heisig5_source,
    korean_ed,
    korean_names,
], ignore_index=True)
display(source)

,codepoint,source,source_region
0,30340,hsk_trad,China
1,20102,hsk_trad,China
2,30637,hsk_trad,China
3,22312,hsk_trad,China
4,26159,hsk_trad,China
...,...,...,...
28162,40860,KoreanName,Korea
28163,40861,KoreanName,Korea
28164,40864,KoreanName,Korea
28165,40866,KoreanName,Korea


In [167]:
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    
    source.to_sql('_temp_char', conn, if_exists='replace', index=False)
    
    conn.execute("""
        INSERT OR IGNORE INTO character_source
        SELECT * FROM _temp_char
    """)
    
    conn.execute("DROP TABLE _temp_char")
    conn.commit()
    
    display(pd.read_sql_query("SELECT * FROM character_source", conn))

,codepoint,source,source_region
0,30340,hsk_trad,China
1,20102,hsk_trad,China
2,30637,hsk_trad,China
3,22312,hsk_trad,China
4,26159,hsk_trad,China
...,...,...,...
28162,40860,KoreanName,Korea
28163,40861,KoreanName,Korea
28164,40864,KoreanName,Korea
28165,40866,KoreanName,Korea


In [168]:
# Unihan Dictionary Indexes
unihan_dic_index = unihan_data[['codepoint', 'kHanYu' ,'kMorohashi', 'kNelson','kKangXi','kIRGKangXi','kDaeJaweon']].copy() 
unihan_dic_index.dropna(subset=['kHanYu' ,'kMorohashi', 'kNelson','kKangXi','kIRGKangXi','kDaeJaweon'], how='all', inplace=True)

display(unihan_dic_index)

,codepoint,kHanYu,kMorohashi,kNelson,kKangXi,kIRGKangXi,kDaeJaweon
0,131072,10011.010,00004 00004:E0100 00005:E0101 00098:E0102,NaN,0075.060,0075.060,NaN
1,131073,10004.020,00009,NaN,0076.021,0076.021,NaN
2,131074,NaN,NaN,NaN,0076.021,0076.021,NaN
3,131075,10008.010,00016,NaN,0076.140,0076.140,NaN
4,131076,NaN,NaN,NaN,0076.141,0076.141,NaN
...,...,...,...,...,...,...,...
102820,64038,NaN,NaN,4770,NaN,NaN,NaN
102821,64039,NaN,NaN,NaN,1308.261,1308.261,NaN
102822,64040,NaN,NaN,NaN,1313.051,1313.051,NaN
102823,64041,NaN,NaN,NaN,1359.101,1359.101,NaN


In [169]:
# Pull the definitive codepoint list from Character_Table (already in the DB).
# Left-joining from this list ensures Dic_Index_Table only contains codepoints
# that are in scope — codepoints in Unihan but not in any of our source lists are excluded.
# Characters with no dictionary index entries get NULL for all index columns.



with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    codepoint_list = pd.read_sql_query("SELECT codepoint FROM character_table", conn)

merged_dic = pd.merge(codepoint_list,unihan_dic_index,on='codepoint',how='left')

In [170]:
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS dic_index_table (
            codepoint INTEGER,
            hanyu INTEGER,
            morohashi INTEGER,
            nelson INTEGER,
            kang_xi INTEGER,
            irg_kang_xi INTEGER,
            dae_jaweon INTEGER,
            CONSTRAINT pk_j_dic_index PRIMARY KEY (codepoint),
            CONSTRAINT fk_codepoint FOREIGN KEY (codepoint)
            REFERENCES character_table (codepoint)
        )
    """)
    conn.commit()

In [171]:
# Populating
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    merged_dic.to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO dic_index_table (codepoint, hanyu, morohashi, nelson, kang_xi, irg_kang_xi, dae_jaweon)
        SELECT codepoint, kHanYu, kMorohashi, kNelson, kKangXi, kIRGKangXi, kDaeJaweon
        FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM dic_index_table", conn))

,codepoint,hanyu,morohashi,nelson,kang_xi,irg_kang_xi,dae_jaweon
0,13466,10231.08,1222,None,119.32,119.32,NaN
1,13527,10300.23,01707:E0101,None,133.33,133.33,NaN
2,13589,10367.07,2305,None,146.2,146.20,NaN
3,13601,10376.13,2405,None,148.28,148.28,NaN
4,13823,10695.01,4395,None,210.23,210.23,NaN
...,...,...,...,...,...,...,...
9835,173668,74800.21,48772,None,1536.07,1536.07,NaN
9836,180501,None,None,None,None,NaN,NaN
9837,182227,None,None,None,None,NaN,NaN
9838,189801,None,None,None,None,NaN,NaN


## Mainland Chinese Simplification Map (`Trad_Simp_Map`)

The PRC's Character Simplification Campaign (1950s–1960s) reduced stroke counts for hundreds of 
characters, creating a parallel simplified script used on the Mainland while Taiwan, Hong Kong, 
and the overseas diaspora retained traditional forms.

The mapping is **many-to-one**: multiple traditional characters can simplify to the same character 
(e.g. 發 and 髮 both simplify to 发). This is why the HSK explode step earlier created one row 
per traditional form. Each traditional codepoint has its own mapping record.

**`Trad_Simp_Map` schema:**
- `Tradcodepoint` — traditional character's codepoint
- `Simpcodepoint` — simplified character's codepoint
- `Trad` — traditional character literal
- `Simp` — simplified character literal
- PRIMARY KEY: `(Tradcodepoint, Simpcodepoint)`, composite, because the mapping can be many-to-many in theory

Note: Both simplified (HSK_Simp) and traditional (HSK_Trad) characters were inserted into Character_Table earlier in this notebook, so both foreign key constraints are valid and enforced.


In [175]:
# Reorder columns in hsk_data_t so they are in a consistent, readable order:
#   t_codepoint | hanzi_trad_list | s_codepoint | hanzi_sc
# The INSERT statement below names columns explicitly, so positional order
# does not affect correctness — this reorder is for readability only.

col = hsk_data_t.pop('hanzi_sc')
hsk_data_t.insert(3,'hanzi_sc',col)

hsk_data_t = hsk_data_t[hsk_data_t['t_codepoint'] != hsk_data_t['s_codepoint']]
display(hsk_data_t)

,t_codepoint,s_codepoint,hanzi_trad_list,hanzi_sc
2,30637,20102,瞭,了
9,36889,36825,這,这
10,20491,20010,個,个
17,35498,35828,說,说
18,20497,20204,們,们
...,...,...,...,...
3149,28396,27818,滬,沪
3151,21555,21556,吳,吴
3155,27472,27431,歐,欧
3160,37159,37011,鄧,邓


In [173]:
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("""
    CREATE TABLE IF NOT EXISTS trad_simp_map (
        trad_codepoint INTEGER,
        simp_codepoint INTEGER,
        trad_literal TEXT,
        simp_literal TEXT,
        CONSTRAINT pk_trad_simp_map PRIMARY KEY (trad_codepoint,simp_codepoint),
        CONSTRAINT fk_trad_codepoint FOREIGN KEY (trad_codepoint)
        REFERENCES character_table (codepoint),
        CONSTRAINT fk_simpcodepoint FOREIGN KEY (simp_codepoint)
        REFERENCES character_table (codepoint)
    )
""")
    conn.commit()

In [174]:
# Populating
with sqlite3.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    hsk_data_t.to_sql('_temp_char', conn, if_exists='replace', index=False)
    conn.execute("""
        INSERT OR IGNORE INTO trad_simp_map (trad_codepoint, trad_literal, simp_codepoint, simp_literal)
        SELECT t_codepoint, hanzi_trad_list, s_codepoint, hanzi_sc
        FROM _temp_char
    """)

    conn.execute("DROP TABLE _temp_char")
    conn.commit()
    display(pd.read_sql_query("SELECT * FROM trad_simp_map", conn))

,trad_codepoint,simp_codepoint,trad_literal,simp_literal
0,30637,20102,瞭,了
1,36889,36825,這,这
2,20491,20010,個,个
3,35498,35828,說,说
4,20497,20204,們,们
...,...,...,...,...
1205,28396,27818,滬,沪
1206,21555,21556,吳,吴
1207,27472,27431,歐,欧
1208,37159,37011,鄧,邓
